In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/samplesubmission.csv
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_262.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_086.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_097.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_157.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_240.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_324.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_034.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_271.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_122.jpg
/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test/test_047.jpg
/kaggle/input/comp

# Prapemrosesan Data & Augmentasi Tingkat Lanjut

In [2]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

# ==========================================
# 1. KONFIGURASI JALUR & HIPERPARAMETER
# ==========================================
BASE_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
TRAIN_DIR = os.path.join(BASE_PATH, "train")
TEST_DIR = os.path.join(BASE_PATH, "test")

# Swin Transformer standar biasanya menggunakan resolusi 224x224
IMG_SIZE = 224 
BATCH_SIZE = 32

# ==========================================
# 2. PEMBUATAN DATAFRAME & STRATIFIED SPLIT
# ==========================================
# Membaca folder train dan memetakan nama kelas ke angka (0-5)
class_names = sorted(os.listdir(TRAIN_DIR))
label2id = {class_name: i for i, class_name in enumerate(class_names)}
id2label = {i: class_name for class_name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

# Membagi data latih menjadi Train (80%) dan Validasi (20%) secara seimbang (Stratified)
train_df, valid_df = train_test_split(
    df_all, test_size=0.2, stratify=df_all['label'], random_state=42
)

# ==========================================
# 3. PIPELINE AUGMENTASI (ALBUMENTATIONS)
# ==========================================
train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    # Mengubah sedikit kecerahan/kontras untuk mensimulasikan cahaya layar
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    # Rotasi dan pergeseran ringan
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    # Sangat penting untuk Swin: Menghapus acak bagian gambar agar model melihat konteks global
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.5),
    # Normalisasi standar ImageNet yang diwajibkan oleh model pre-trained timm
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 4. KELAS PYTORCH DATASET
# ==========================================
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        # Membaca gambar menggunakan OpenCV (format BGR)
        image = cv2.imread(img_path)
        # Ubah ke format RGB karena model dilatih dengan gambar RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        # Pastikan label bertipe long integer untuk fungsi Loss PyTorch
        return image, torch.tensor(label, dtype=torch.long)

# ==========================================
# 5. INISIALISASI DATALOADER
# ==========================================
train_dataset = SpoofingDataset(train_df, transforms=train_transforms)
valid_dataset = SpoofingDataset(valid_df, transforms=valid_transforms)

# Parameter num_workers disesuaikan dengan CPU di Kaggle Notebook (biasanya 2 atau 4 aman)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("✅ Prapemrosesan Selesai!")
print(f"Total Data Latih    : {len(train_dataset)} gambar")
print(f"Total Data Validasi : {len(valid_dataset)} gambar")
print(f"Kamus Label         : {id2label}")

✅ Prapemrosesan Selesai!
Total Data Latih    : 1321 gambar
Total Data Validasi : 331 gambar
Kamus Label         : {0: 'fake_mannequin', 1: 'fake_mask', 2: 'fake_printed', 3: 'fake_screen', 4: 'fake_unknown', 5: 'realperson'}


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_24/829151087.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.5),


# Membangun Arsitektur Swin

In [3]:
import torch
import torch.nn as nn
# Pastikan Anda sudah menginstal timm di Kaggle (biasanya sudah bawaan)
# Jika belum, jalankan: !pip install timm
import timm

# ==========================================
# 1. KONFIGURASI PERANGKAT (GPU / CPU)
# ==========================================
# Sangat penting untuk memindahkan model dan data ke GPU agar pelatihan cepat
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# ==========================================
# 2. INISIALISASI MODEL SWIN TRANSFORMER
# ==========================================
# Kita gunakan versi 'base' dengan ukuran patch 4 dan window 7 untuk resolusi 224x224.
# Jika memori GPU Kaggle Anda penuh (Out of Memory), Anda bisa menurunkan ke 'swin_tiny_patch4_window7_224'
MODEL_NAME = 'swin_base_patch4_window7_224.ms_in22k'
NUM_CLASSES = 6 # Sesuai jumlah kategori kompetisi Anda

print(f"Mengunduh dan membangun model: {MODEL_NAME}...")

# Kehebatan timm: Kita cukup memasukkan num_classes=6, 
# dan timm akan otomatis membuang head asli (untuk ribuan kelas) 
# dan menggantinya dengan Linear Layer baru untuk 6 kelas!
model = timm.create_model(
    MODEL_NAME, 
    pretrained=True, 
    num_classes=NUM_CLASSES
)

# Pindahkan model ke GPU
model = model.to(device)

# ==========================================
# 3. UJI COBA MODEL (SANITY CHECK)
# ==========================================
# Kita buat tensor acak (dummy data) yang mensimulasikan 1 batch berisi 4 gambar (ukuran 224x224 RGB)
dummy_input = torch.randn(4, 3, 224, 224).to(device)

# Lewatkan data ke dalam model
with torch.no_grad():
    dummy_output = model(dummy_input)

print("✅ Model berhasil dibangun!")
print(f"Bentuk input (Batch, Channel, Height, Width) : {dummy_input.shape}")
print(f"Bentuk output (Batch, Num_Classes)           : {dummy_output.shape}")

Menggunakan perangkat: cuda
Mengunduh dan membangun model: swin_base_patch4_window7_224.ms_in22k...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

✅ Model berhasil dibangun!
Bentuk input (Batch, Channel, Height, Width) : torch.Size([4, 3, 224, 224])
Bentuk output (Batch, Num_Classes)           : torch.Size([4, 6])


# Strategi Pelatihan (Warm-up & Loss)

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import f1_score
from tqdm import tqdm
import numpy as np

# ==========================================
# 1. DEFINISI FOCAL LOSS UNTUK KELAS IMBALANS
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma # Parameter fokus, standar 2
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Hitung Cross Entropy Loss mentah
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        # Dapatkan probabilitas (pt) dari tebakan target
        pt = torch.exp(-ce_loss)
        # Terapkan formula Focal Loss
        F_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        return F_loss

# ==========================================
# 2. INISIALISASI OPTIMIZER, LOSS, & SCHEDULER
# ==========================================
EPOCHS = 10  # Mulai dengan 10 epoch untuk uji coba
LEARNING_RATE = 2e-5 # Learning rate kecil sangat disarankan untuk Transformer
PATIENCE = 3 # Untuk Early Stopping

criterion = FocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
# Menurunkan learning rate secara mulus hingga titik minimum
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ==========================================
# 3. FUNGSI PELATIHAN (TRAINING LOOP)
# ==========================================
best_val_f1 = 0.0
patience_counter = 0

print("🚀 Memulai Pelatihan Swin Transformer...")

for epoch in range(EPOCHS):
    # --- FASE TRAINING ---
    model.train()
    train_loss = 0.0
    
    # Gunakan tqdm untuk progress bar yang rapi
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad() # Bersihkan gradien sisa
        outputs = model(images) # Lakukan prediksi maju (Forward Pass)
        loss = criterion(outputs, labels) # Hitung kerugian (Loss)
        
        loss.backward() # Hitung gradien (Backward Pass)
        optimizer.step() # Perbarui bobot model
        
        train_loss += loss.item() * images.size(0)
        train_bar.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader.dataset)
    
    # --- FASE VALIDASI ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    
    val_bar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
    with torch.no_grad(): # Matikan perhitungan gradien agar hemat memori
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            
            # Ambil kelas dengan probabilitas tertinggi (argmax)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(valid_loader.dataset)
    
    # Menghitung Macro F1-Score (Sesuai metrik kompetisi DAC Find IT!)
    val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"\n📊 Hasil Epoch {epoch+1}:")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
    
    # Perbarui Scheduler
    scheduler.step()
    
    # --- SIMPAN MODEL TERBAIK & EARLY STOPPING ---
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(model.state_dict(), 'best_swin_model.pth')
        print(f"🌟 Model membaik! Disimpan ke 'best_swin_model.pth'")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"⚠️ Model tidak membaik. Kesabaran: {patience_counter}/{PATIENCE}")
        
    if patience_counter >= PATIENCE:
        print("\n🛑 Early Stopping dipicu! Pelatihan dihentikan untuk mencegah overfitting.")
        break

print(f"\n✅ Pelatihan Selesai! Skor Macro F1 Validasi Terbaik: {best_val_f1:.4f}")

🚀 Memulai Pelatihan Swin Transformer...


Epoch 1/10 [Valid]: 100%|██████████| 11/11 [00:07<00:00,  1.54it/s]



📊 Hasil Epoch 1:
Train Loss: 0.7464 | Val Loss: 0.4598 | Val Macro F1: 0.7312
🌟 Model membaik! Disimpan ke 'best_swin_model.pth'


Epoch 2/10 [Valid]: 100%|██████████| 11/11 [00:05<00:00,  1.85it/s]



📊 Hasil Epoch 2:
Train Loss: 0.3612 | Val Loss: 0.4076 | Val Macro F1: 0.7636
🌟 Model membaik! Disimpan ke 'best_swin_model.pth'


...


📊 Hasil Epoch 4:
Train Loss: 0.1713 | Val Loss: 0.3913 | Val Macro F1: 0.7662
⚠️ Model tidak membaik. Kesabaran: 1/3


Epoch 5/10 [Valid]: 100%|██████████| 11/11 [00:05<00:00,  1.85it/s]



📊 Hasil Epoch 5:
Train Loss: 0.1369 | Val Loss: 0.4028 | Val Macro F1: 0.7699
⚠️ Model tidak membaik. Kesabaran: 2/3


Epoch 6/10 [Valid]: 100%|██████████| 11/11 [00:06<00:00,  1.71it/s]


📊 Hasil Epoch 6:
Train Loss: 0.1145 | Val Loss: 0.4044 | Val Macro F1: 0.7571
⚠️ Model tidak membaik. Kesabaran: 3/3

🛑 Early Stopping dipicu! Pelatihan dihentikan untuk mencegah overfitting.

✅ Pelatihan Selesai! Skor Macro F1 Validasi Terbaik: 0.7881


# Kalibrasi Probabilitas (Thresholding)

In [5]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score
from scipy.optimize import minimize
import warnings

# Mengabaikan warning scipy agar output terminal tetap bersih
warnings.filterwarnings("ignore")

# ==========================================
# 1. MUAT MODEL TERBAIK & PERSIAPAN DATA
# ==========================================
print("🔄 Memuat bobot model terbaik...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load('best_swin_model.pth', map_location=device))
model.eval()

all_probs = []
all_labels = []

print("🔍 Mengekstrak probabilitas mentah dari Set Validasi...")
with torch.no_grad():
    for images, labels in valid_loader: # Menggunakan valid_loader dari Langkah 1
        images = images.to(device)
        outputs = model(images)
        # Ubah output logits (mentah) menjadi probabilitas 0.0 - 1.0
        probs = F.softmax(outputs, dim=1) 
        
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# ==========================================
# 2. EVALUASI SEBELUM KALIBRASI
# ==========================================
preds_before = np.argmax(all_probs, axis=1)
f1_before = f1_score(all_labels, preds_before, average='macro')
print(f"📊 Macro F1-Score SEBELUM Kalibrasi : {f1_before:.5f}")

# ==========================================
# 3. PROSES OPTIMASI THRESHOLD
# ==========================================
# Kita membuat fungsi yang akan mencari nilai 'F1 terburuk' (karena scipy mencari nilai minimum)
# yang nantinya dibalik menjadi F1 maksimal.
def optimize_f1(weights):
    # Kalikan probabilitas model dengan bobot tebakan algoritma
    weighted_probs = all_probs * weights
    # Ambil kelas dengan probabilitas tertinggi setelah dikalikan bobot
    calibrated_preds = np.argmax(weighted_probs, axis=1)
    # Kembalikan nilai negatif F1 (agar scipy.minimize bisa meminimalkannya)
    return -1.0 * f1_score(all_labels, calibrated_preds, average='macro')

# Inisialisasi bobot awal (semua kelas dikalikan 1.0 / tidak diubah)
initial_weights = [1.0] * 6 # 6 adalah jumlah kelas kita

print("⚙️ Memulai optimasi kalibrasi probabilitas (Nelder-Mead)...")
# Algoritma Nelder-Mead sangat bagus untuk fungsi diskrit seperti F1-Score
result = minimize(optimize_f1, initial_weights, method='Nelder-Mead')

best_weights = result.x

# ==========================================
# 4. EVALUASI SETELAH KALIBRASI
# ==========================================
weighted_probs_after = all_probs * best_weights
preds_after = np.argmax(weighted_probs_after, axis=1)
f1_after = f1_score(all_labels, preds_after, average='macro')

print(f"🚀 Macro F1-Score SETELAH Kalibrasi : {f1_after:.5f}")
print(f"📈 Peningkatan Skor                 : {f1_after - f1_before:.5f}")
print("\n✅ Simpan array bobot ini untuk dipakai di Test Set (Langkah 5):")
print(f"best_weights = {list(best_weights)}")

🔄 Memuat bobot model terbaik...
🔍 Mengekstrak probabilitas mentah dari Set Validasi...
📊 Macro F1-Score SEBELUM Kalibrasi : 0.78808
⚙️ Memulai optimasi kalibrasi probabilitas (Nelder-Mead)...
🚀 Macro F1-Score SETELAH Kalibrasi : 0.79706
📈 Peningkatan Skor                 : 0.00897

✅ Simpan array bobot ini untuk dipakai di Test Set (Langkah 5):
best_weights = [np.float64(1.025), np.float64(1.025), np.float64(0.8999999999999999), np.float64(1.025), np.float64(1.025), np.float64(1.025)]


# Prediksi TTA & Pengumpulan (Submission)

In [6]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# ==========================================
# 1. PERSIAPAN JALUR & TEMPLATE SUBMISSION
# ==========================================
TEST_DIR = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/samplesubmission.csv"

sub_df = pd.read_csv(SAMPLE_SUB_PATH)

class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: class_name for i, class_name in enumerate(class_names)}

test_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 2. DATASET ANTI-CRASH (DETEKSI OTOMATIS)
# ==========================================
class TestDatasetTTA(Dataset):
    def __init__(self, df, test_dir, transforms):
        self.df = df
        self.test_dir = test_dir
        self.transforms = transforms
        
        # PETA CERDAS: Mencatat semua file yang benar-benar ada di folder test
        # Format: {'test_001': 'test_001.png', 'test_002': 'test_002.jpg', ...}
        self.existing_files = {}
        if os.path.exists(test_dir):
            for f in os.listdir(test_dir):
                name_without_ext = os.path.splitext(f)[0]
                self.existing_files[name_without_ext] = f

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Ambil ID mentah dari CSV
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] # Bersihkan jika ada ekstensi bawaan
        
        image = None
        
        # Cek apakah file ada di peta cerdas kita
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) # Coba baca file
            
        # Jika file tidak ada ATAU rusak (OpenCV mengembalikan None)
        if image is None:
            # Buat gambar hitam kosong (dummy) agar sistem tidak crash
            image = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            # Jika berhasil dibaca, ubah ke RGB
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # 1. Gambar Asli
        augmented_orig = self.transforms(image=image)
        img_orig = augmented_orig['image']
        
        # 2. Gambar TTA (Dibalik)
        image_flipped = cv2.flip(image, 1) 
        augmented_flip = self.transforms(image=image_flipped)
        img_flip = augmented_flip['image']

        return img_id, img_orig, img_flip

test_dataset = TestDatasetTTA(sub_df, TEST_DIR, transforms=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# ==========================================
# 3. EKSEKUSI PREDIKSI (INFERENCE)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# Bobot hasil kalibrasi Anda (gunakan [1.0] * 6 jika tidak memakai kalibrasi)
best_weights = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0]) 

all_predictions = []

print(f"🚀 Memulai prediksi {len(sub_df)} data (Sistem Anti-Crash Aktif)...")

with torch.no_grad():
    for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc="Menebak Data"):
        imgs_orig = imgs_orig.to(device)
        imgs_flip = imgs_flip.to(device)

        out_orig = model(imgs_orig)
        prob_orig = F.softmax(out_orig, dim=1)

        out_flip = model(imgs_flip)
        prob_flip = F.softmax(out_flip, dim=1)

        prob_avg = (prob_orig + prob_flip) / 2.0
        prob_avg = prob_avg.cpu().numpy()

        weighted_probs = prob_avg * best_weights
        final_preds = np.argmax(weighted_probs, axis=1)

        all_predictions.extend(final_preds)

# ==========================================
# 4. PENYIMPANAN KE TEMPLATE
# ==========================================
print("📝 Menyusun file submission...")

# Ubah angka menjadi teks label
final_labels = [id2label[pred] for pred in all_predictions]

# Langsung timpa kolom 'label' di template asli
sub_df['label'] = final_labels

# Simpan ke CSV
sub_df.to_csv('submission.csv', index=False)

print("✅ Selesai! File 'submission.csv' berhasil dibuat tanpa error.")
print(sub_df.head())

🚀 Memulai prediksi 404 data (Sistem Anti-Crash Aktif)...


Menebak Data: 100%|██████████| 13/13 [00:13<00:00,  1.00s/it]

📝 Menyusun file submission...
✅ Selesai! File 'submission.csv' berhasil dibuat tanpa error.
         id         label
0  test_001   fake_screen
1  test_002   fake_screen
2  test_003     fake_mask
3  test_004    realperson
4  test_005  fake_printed


In [7]:
#0,65078